# Music Annotation Project V9.0B
## Frozen V8.8 transfer to 151 real Notebook 06 to 07 human-reviewed masks

**Runtime: CPU. Run all in Colab.**

**Hypothesis.** With the human-reviewed corrected mask retained as authority, frozen V8.8 transfers its correction performance and safety to the operational HITL population. Remaining failures are associated with measurable properties of the human mask edit.

**Guardrails:** no retraining, recalibration, threshold movement or raw-YOLO authority claim; clean truth is loaded only after inference outputs and mask-edit predictors are locked; outside-mask pixels must remain immutable; failure is fail-closed and auditable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scipy scikit-learn statsmodels pyarrow matplotlib joblib

## 1. Configuration and predeclared endpoints

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,math,shutil,zipfile,importlib.util,sys,platform
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt,joblib
from sklearn.model_selection import GroupKFold,cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score,average_precision_score,brier_score_loss
from sklearn.inspection import permutation_importance
from statsmodels.stats.proportion import proportion_confint
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N53_ROOT=ROOT/'observed_print_classifier_v8_8_notebook53R3'
HITL=ROOT/'human_in_the_middle'; DRAFTS=HITL/'draft_masks'; DRAFT_AUDIT=HITL/'draft_audit'; REVIEW_AUDIT=HITL/'audit'
CORRECTED=ROOT/'masks'/'corrected'; CLEAN=ROOT/'source'/'clean'; ALIGNED=ROOT/'aligned'
OUT=ROOT/'v9_0b_hitl_mask_transfer'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_v9_0b')
I=OUT/'01_inputs';E=OUT/'02_engine';L=OUT/'03_lock';T=OUT/'04_tables';V=OUT/'05_visuals';P=OUT/'06_pages';A=OUT/'07_audit';U=OUT/'upload'
for p in [OUT,I,E,L,T,V,P,A,U]:p.mkdir(parents=True,exist_ok=False)
EXPECTED=151;SEED=42;BOOT=10000;MIN_AREA=20
STRUCT={'ink_recall':.985,'ink_f1':.95,'edge_f1':.90,'removed_print':.001,'mae':16.0}
print(OUT)

## 2. Helpers and frozen-engine resolution

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def imap(d):return {p.stem:p for p in Path(d).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(d).exists() else {}
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def ah(a):
 a=np.ascontiguousarray(a);return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+a.tobytes()).hexdigest()
def atomic(p,x):Path(p).write_text(json.dumps(x,indent=2,sort_keys=True,default=str))
def latest(root,req):
 q=[p for p in Path(root).iterdir() if p.is_dir() and all((p/r).exists() for r in req)] if Path(root).exists() else []
 return max(q,key=lambda p:p.stat().st_mtime) if q else None
def fail(stage,msg,detail=None):atomic(A/f'failure_{stage}.json',{'stage':stage,'message':msg,'detail':detail});raise RuntimeError(msg)
def wilson(e,n):return float(proportion_confint(e,n,method='wilson')[1]) if n else np.nan
N53=latest(N53_ROOT,['stage_f_verdict.json','experiment_identity.json','tables/page_equivalence.csv','pages'])
if N53 is None:fail('n53','No complete Notebook 53R3 run')
id53=json.loads((N53/'experiment_identity.json').read_text());v53=json.loads((N53/'stage_f_verdict.json').read_text())
if id53.get('mode')!='REGISTERED_REPLAY' or not str(id53.get('version','')).startswith('53R3-v8.8b') or 'PASS' not in str(v53.get('verdict','')):fail('n53_identity','Frozen V8.8 identity/verdict invalid',{'identity':id53,'verdict':v53})
runner=next((p for p in [N53/'frozen_v88_runner.py',N53/'runtime'/'frozen_v88_runner.py',N53/'operational_builder.py'] if p.exists()),None)
if runner is None:fail('runner','Notebook 53R3 must export frozen_v88_runner.py with run_page().')
s=importlib.util.spec_from_file_location('v88',runner);v88=importlib.util.module_from_spec(s);s.loader.exec_module(v88)
if not hasattr(v88,'run_page'):fail('runner_contract','run_page missing')
atomic(E/'engine.json',{'run':str(N53),'version':id53.get('version'),'mode':id53.get('mode'),'runner':str(runner),'runner_sha256':sha(runner)})

## 3. Resolve the real Notebook 06 to 07 roster

In [ ]:
def audits(d):
 out=[]
 for p in Path(d).glob('*.json'):
  try:q=json.loads(p.read_text());q['_path']=str(p);out.append(q)
  except Exception as e:fail('audit_parse',str(e),str(p))
 return out
def stem(r,keys):
 for k in keys:
  if r.get(k):return Path(str(r[k])).stem
 return None
def existing(r,keys):
 for k in keys:
  if r.get(k) and Path(str(r[k])).exists():return Path(str(r[k]))
 return None
DA,RA=audits(DRAFT_AUDIT),audits(REVIEW_AUDIT);cm,dm,cl,al=imap(CORRECTED),imap(DRAFTS),imap(CLEAN),imap(ALIGNED)
rows=[];bad=[]
for page,cp in sorted(cm.items()):
 rr=[r for r in RA if stem(r,['corrected_mask_output','mask_output'])==page]
 if len(rr)!=1:bad.append({'page':page,'reason':'review_audit_count','count':len(rr)});continue
 r=rr[0]; h=str(r.get('source_md5','')).lower(); dd=[d for d in DA if h and str(d.get('source_md5','')).lower()==h]
 if len(dd)!=1:dd=[d for d in DA if stem(d,['draft_mask','source_file'])==page]
 if len(dd)!=1:bad.append({'page':page,'reason':'draft_audit_count','count':len(dd)});continue
 d=dd[0]; dp=existing(d,['draft_mask','draft_mask_output']) or dm.get(page); ap=existing(r,['aligned_output','aligned_file']) or al.get(page); kp=cl.get(page); conf=existing(d,['confidence_map','confidence_map_output'])
 if not all([dp,ap,kp]):bad.append({'page':page,'reason':'missing_image'});continue
 rows.append({'page':page,'draft':str(dp),'corrected':str(cp),'observed':str(ap),'clean':str(kp),'confidence':str(conf) if conf else None,'draft_audit':d['_path'],'review_audit':r['_path']})
roster=pd.DataFrame(rows).sort_values('page');pd.DataFrame(bad).to_csv(A/'unresolved.csv',index=False);roster.to_csv(I/'hitl_roster.csv',index=False)
if len(roster)!=EXPECTED:fail('roster',f'Expected {EXPECTED}, resolved {len(roster)}',{'unresolved':len(bad)})
print('Roster',len(roster))

## 4. Mask-edit characteristics

In [ ]:
def binm(a):return a>0
def comps(m,min_area=1):
 n,_,s,_=cv2.connectedComponentsWithStats(m.astype(np.uint8),8);return sum(s[i,4]>=min_area for i in range(1,n))
def perim(m):
 c,_=cv2.findContours(m.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_NONE);return sum(cv2.arcLength(x,True) for x in c)
def features(d,c,conf):
 add=c&~d;era=d&~c;inter=d&c;union=d|c;xor=add|era;dep=cv2.distanceTransform(c.astype(np.uint8),cv2.DIST_L2,5)
 return {'draft_pixels':d.sum(),'corrected_pixels':c.sum(),'added_pixels':add.sum(),'erased_pixels':era.sum(),'xor_pixels':xor.sum(),'mask_iou':inter.sum()/max(union.sum(),1),'addition_fraction_final':add.sum()/max(c.sum(),1),'erasure_fraction_draft':era.sum()/max(d.sum(),1),'net_area_change_fraction':(c.sum()-d.sum())/max(d.sum(),1),'draft_components':comps(d,MIN_AREA),'corrected_components':comps(c,MIN_AREA),'component_count_change':comps(c,MIN_AREA)-comps(d,MIN_AREA),'perimeter_change_fraction':(perim(c)-perim(d))/max(perim(d),1),'edit_fragmentation':comps(xor,1),'maximum_mask_depth':dep[c].max() if c.any() else 0,'mean_mask_depth':dep[c].mean() if c.any() else 0,'confidence_mean_final':conf[c].mean() if conf is not None and c.any() else np.nan,'confidence_mean_added':conf[add].mean() if conf is not None and add.any() else np.nan,'confidence_mean_erased':conf[era].mean() if conf is not None and era.any() else np.nan,'confidence_low_fraction_final':(conf[c]<.5).mean() if conf is not None and c.any() else np.nan}
loaded={};fr=[];hashes=[]
for _,r in roster.iterrows():
 o=cv2.imread(r.observed,0);k=cv2.imread(r.clean,0);d=cv2.imread(r.draft,0);c=cv2.imread(r.corrected,0)
 if any(x is None for x in [o,k,d,c]) or len({x.shape for x in [o,k,d,c]})!=1:fail('shape',r.page)
 conf=np.load(r.confidence).astype(np.float32) if isinstance(r.confidence,str) and Path(r.confidence).exists() else None
 if conf is not None and conf.shape!=o.shape:fail('confidence_shape',r.page)
 d,c=binm(d),binm(c);q={'page':r.page};q.update(features(d,c,conf));fr.append(q);loaded[r.page]={'obs':o,'clean':k,'draft':d,'mask':c,'conf':conf}
 hashes.append({'page':r.page,'observed':sha(r.observed),'clean':sha(r.clean),'draft':sha(r.draft),'corrected':sha(r.corrected),'confidence':sha(r.confidence) if isinstance(r.confidence,str) and Path(r.confidence).exists() else None})
feat=pd.DataFrame(fr);feat.to_csv(T/'mask_edit_features.csv',index=False);pd.DataFrame(hashes).to_csv(A/'input_hashes.csv',index=False)

## 5. Frozen V8.8 inference and prediction lock

In [ ]:
outs=[]
for page,z in loaded.items():
 d=P/page;d.mkdir();res=v88.run_page(observed=z['obs'],authoritative_mask=(z['mask'].astype(np.uint8)*255),page_id=page,output_dir=d)
 req={'corrected','m3','m5_remove','safe_remove','safe_remove_probability','reason_map'}
 if not req.issubset(res):fail('runner_result',page,sorted(req-set(res)))
 auth=z['mask'];out=res['corrected'];outside=int((out[~auth]!=z['obs'][~auth]).sum())
 if outside:fail('outside_authority',page,outside)
 cv2.imwrite(str(d/'observed.png'),z['obs']);cv2.imwrite(str(d/'mask.png'),auth.astype(np.uint8)*255);cv2.imwrite(str(d/'corrected.png'),out)
 np.savez_compressed(d/'arrays.npz',**{k:np.asarray(v) for k,v in res.items() if isinstance(v,np.ndarray)})
 outs.append({'page':page,'outside_drift':outside,'corrected_hash':ah(out),'safe_remove_pixels':int(np.asarray(res['safe_remove']).astype(bool).sum())})
pd.DataFrame(outs).to_csv(A/'page_outputs.csv',index=False)
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0B','engine':json.loads((E/'engine.json').read_text()),'roster_sha256':sha(I/'hitl_roster.csv'),'features_sha256':sha(T/'mask_edit_features.csv'),'outputs_sha256':sha(A/'page_outputs.csv'),'arrays':{p:sha(P/p/'arrays.npz') for p in sorted(loaded)}}
lock['lock_id']=hashlib.sha256(json.dumps(lock,sort_keys=True).encode()).hexdigest();atomic(L/'prediction_lock.json',lock);print(lock['lock_id'])

## 6. Clean-truth scoring after lock

In [ ]:
def ink(a):return cv2.adaptiveThreshold(a,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edge(a):return cv2.Canny(a,70,170)>0
def tf1(a,b,m):
 a=a&m;b=b&m
 if not a.any():return np.nan,np.nan,np.nan
 k=np.ones((3,3),np.uint8);p=(b&(cv2.dilate(a.astype(np.uint8),k)>0)).sum()/max(b.sum(),1);r=(a&(cv2.dilate(b.astype(np.uint8),k)>0)).sum()/a.sum();return p,r,2*p*r/max(p+r,1e-12)
def score(clean,recon,m,remove):
 ci,ri=ink(clean),ink(recon);ip,ir,if1=tf1(ci,ri,m);ep,er,ef1=tf1(edge(clean),edge(recon),m);diff=np.abs(clean.astype(float)-recon.astype(float));mae=diff[m].mean();rp=(remove&ci&m).sum()/max(m.sum(),1)
 fail=[]
 if np.isfinite(ir) and ir<STRUCT['ink_recall']:fail.append('ink_recall')
 if np.isfinite(if1) and if1<STRUCT['ink_f1']:fail.append('ink_f1')
 if np.isfinite(ef1) and ef1<STRUCT['edge_f1']:fail.append('edge_f1')
 if rp>STRUCT['removed_print']:fail.append('removed_print')
 if mae>STRUCT['mae']:fail.append('mae')
 return {'mae':mae,'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'removed_clean_print_fraction':rp,'structural_safe':not fail,'failure_reasons':'|'.join(fail)}
ps=[];cs=[]
for page,z in loaded.items():
 out=cv2.imread(str(P/page/'corrected.png'),0)
 with np.load(P/page/'arrays.npz') as a:remove=a['safe_remove']>0
 q=score(z['clean'],out,z['mask'],remove);q.update({'page':page,'mask_pixels':z['mask'].sum()});ps.append(q)
 n,lab,st,_=cv2.connectedComponentsWithStats(z['mask'].astype(np.uint8),8)
 for i in range(1,n):
  if st[i,4]<MIN_AREA:continue
  q=score(z['clean'],out,lab==i,remove);q.update({'page':page,'component':i,'x':st[i,0],'y':st[i,1],'w':st[i,2],'h':st[i,3],'area':st[i,4]});cs.append(q)
page=pd.DataFrame(ps);comp=pd.DataFrame(cs);page.to_csv(T/'page_outcomes.csv',index=False);comp.to_parquet(T/'component_outcomes.parquet',index=False)

## 7. Transfer result and uncertainty

In [ ]:
unsafe=int((~page.structural_safe).sum());n=len(page)
def boot(metric):
 rng=np.random.default_rng(SEED);vals=[];ids=page.page.to_numpy()
 for _ in range(BOOT):
  q=page.set_index('page').loc[rng.choice(ids,len(ids),replace=True)];vals.append(q[metric].mean())
 return [float(np.mean(vals)),float(np.quantile(vals,.025)),float(np.quantile(vals,.975))]
summary={'pages':n,'safe_pages':n-unsafe,'unsafe_pages':unsafe,'unsafe_rate':unsafe/n,'unsafe_wilson95_upper':wilson(unsafe,n),'outside_authority_drift':int(pd.read_csv(A/'page_outputs.csv').outside_drift.sum())}
for m in ['ink_recall','ink_f1','edge_f1','mae']:summary[m+'_bootstrap']=boot(m)
atomic(OUT/'transfer_summary.json',summary);pd.DataFrame([summary]).to_csv(T/'transfer_summary.csv',index=False);print(json.dumps(summary,indent=2))

## 8. Which mask-edit characteristics predict remaining failures?

In [ ]:
data=feat.merge(page,on='page',validate='one_to_one');data['failure']=~data.structural_safe;data.to_csv(T/'analysis_dataset.csv',index=False)
PRED=['draft_pixels','corrected_pixels','added_pixels','erased_pixels','xor_pixels','mask_iou','addition_fraction_final','erasure_fraction_draft','net_area_change_fraction','draft_components','corrected_components','component_count_change','perimeter_change_fraction','edit_fragmentation','maximum_mask_depth','mean_mask_depth','confidence_mean_final','confidence_mean_added','confidence_mean_erased','confidence_low_fraction_final']
X=data[PRED].replace([np.inf,-np.inf],np.nan);y=data.failure.astype(int);uni=[]
import statsmodels.api as sm
for p in PRED:
 q=pd.DataFrame({'x':pd.to_numeric(X[p],errors='coerce'),'y':y}).dropna();iq=q.x.quantile(.75)-q.x.quantile(.25)
 if len(q)<20 or q.y.nunique()<2 or iq==0:uni.append({'predictor':p,'odds_ratio_iqr':np.nan,'low':np.nan,'high':np.nan,'p':np.nan});continue
 try:
  z=(q.x-q.x.median())/iq;f=sm.Logit(q.y,sm.add_constant(z)).fit(disp=False);b,se=f.params.iloc[1],f.bse.iloc[1];uni.append({'predictor':p,'odds_ratio_iqr':np.exp(b),'low':np.exp(b-1.96*se),'high':np.exp(b+1.96*se),'p':f.pvalues.iloc[1]})
 except Exception as e:uni.append({'predictor':p,'error':str(e)})
pd.DataFrame(uni).to_csv(T/'univariable_predictors.csv',index=False)
models=[];imps=[];oof=[]
if y.nunique()>=2:
 cv=GroupKFold(n_splits=min(10,len(data)))
 for name,m in {'logistic':Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler()),('m',LogisticRegression(class_weight='balanced',max_iter=5000,random_state=SEED))]),'forest':Pipeline([('i',SimpleImputer(strategy='median')),('m',RandomForestClassifier(n_estimators=1000,min_samples_leaf=5,class_weight='balanced_subsample',random_state=SEED,n_jobs=-1))])}.items():
  pr=cross_val_predict(m,X,y,groups=data.page,cv=cv,method='predict_proba')[:,1];models.append({'model':name,'roc_auc':roc_auc_score(y,pr),'average_precision':average_precision_score(y,pr),'brier':brier_score_loss(y,pr),'prevalence':y.mean()});oof.extend({'page':data.iloc[i].page,'model':name,'truth':y.iloc[i],'probability':pr[i]} for i in range(len(y)));m.fit(X,y);q=permutation_importance(m,X,y,n_repeats=50,random_state=SEED,scoring='average_precision');imps.extend({'model':name,'predictor':p,'importance_mean':a,'importance_std':b} for p,a,b in zip(PRED,q.importances_mean,q.importances_std))
pd.DataFrame(models).to_csv(T/'oof_model_summary.csv',index=False);pd.DataFrame(oof).to_csv(T/'oof_predictions.csv',index=False);pd.DataFrame(imps).to_csv(T/'permutation_importance.csv',index=False)

## 9. Failure audit images and verdict

In [ ]:
rank=comp.assign(severity=(~comp.structural_safe).astype(int)*1000+comp.mae).sort_values('severity',ascending=False).head(40)
for j,(_,r) in enumerate(rank.iterrows(),1):
 z=loaded[r.page];out=cv2.imread(str(P/r.page/'corrected.png'),0);m=45;x0=max(0,int(r.x)-m);y0=max(0,int(r.y)-m);x1=min(z['obs'].shape[1],int(r.x+r.w)+m);y1=min(z['obs'].shape[0],int(r.y+r.h)+m)
 ims=[z['obs'][y0:y1,x0:x1],z['draft'][y0:y1,x0:x1],z['mask'][y0:y1,x0:x1],out[y0:y1,x0:x1],z['clean'][y0:y1,x0:x1]];fig,ax=plt.subplots(1,5,figsize=(20,4))
 for a,im,t in zip(ax,ims,['observed','draft','reviewed','V8.8','clean']):a.imshow(im,cmap='gray');a.set_title(t);a.axis('off')
 fig.suptitle(f'{r.page} C{int(r.component)} safe={r.structural_safe} {r.failure_reasons}');plt.tight_layout();plt.savefig(V/f'{j:03d}_{r.page}_c{int(r.component)}.png',dpi=150);plt.close(fig)
model=pd.read_csv(T/'oof_model_summary.csv') if (T/'oof_model_summary.csv').exists() else pd.DataFrame();pred='PREDICTIVE_SIGNAL' if len(model) and model.average_precision.max()>model.prevalence.max()+.05 else 'NO_CLEAR_SIGNAL'
verdict={'verdict':'VALID' if summary['outside_authority_drift']==0 and n==EXPECTED else 'INVALID','pages':n,'unsafe_pages':unsafe,'unsafe_wilson95_upper':summary['unsafe_wilson95_upper'],'predictor_result':pred,'raw_yolo_mask_authority_tested':False,'production_authority':False,'lock_id':lock['lock_id']};atomic(OUT/'v9_0b_verdict.json',verdict)
(OUT/'executive_summary.txt').write_text('
'.join(['V9.0B HITL MASK TRANSFER',f'Pages: {n}',f'Unsafe pages: {unsafe}',f'Wilson upper: {summary["unsafe_wilson95_upper"]:.6f}',f'Predictor result: {pred}','Reviewed masks are authoritative. Raw YOLO mask authority was not tested.']))
print(json.dumps(verdict,indent=2))

## 10. Final Upload Package

In [ ]:
files=[OUT/'executive_summary.txt',OUT/'v9_0b_verdict.json',OUT/'transfer_summary.json',L/'prediction_lock.json',T/'transfer_summary.csv',T/'analysis_dataset.csv',T/'univariable_predictors.csv',T/'oof_model_summary.csv',T/'permutation_importance.csv',A/'input_hashes.csv',A/'page_outputs.csv']+sorted(V.glob('*.png'))[:30]
miss=[str(x) for x in files if not x.exists()]
if miss:fail('upload','Missing upload artefacts',miss)
rows=[]
for x in files:
 d=U/('visuals' if x.suffix=='.png' else 'tables' if x.suffix=='.csv' else 'audit')/x.name;d.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(x,d);rows.append({'file':str(d.relative_to(U)),'bytes':d.stat().st_size,'sha256':sha(d)})
pd.DataFrame(rows).to_csv(U/'upload_manifest.csv',index=False);(U/'README.txt').write_text('V9.0B transfer experiment. Start with executive_summary.txt and v9_0b_verdict.json. Reviewed masks are authoritative.')
zpath=OUT/'60_V9_0B_HITL_Mask_Transfer_upload_bundle.zip'
with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
 for p in U.rglob('*'):
  if p.is_file():z.write(p,p.relative_to(U.parent))
print(zpath,sha(zpath))

## Output order
1. `executive_summary.txt`
2. `v9_0b_verdict.json`
3. `transfer_summary.json`
4. `04_tables/analysis_dataset.csv`
5. `04_tables/univariable_predictors.csv`
6. `04_tables/oof_model_summary.csv`
7. `04_tables/permutation_importance.csv`
8. `05_visuals/`
9. `60_V9_0B_HITL_Mask_Transfer_upload_bundle.zip`

**Next question:** Do the same mask-edit predictors remain stable when the experiment is repeated on a fresh reviewed-mask cohort rather than the 151-page operational population?